# Document Question Answering System (RAG)

This notebook implements a Retrieval-Augmented Generation (RAG) system that answers questions based on a custom PDF document.

Instead of relying only on a language model's internal knowledge, the system retrieves relevant chunks from the document and then generates answers grounded in that information.

**Pipeline:**
1. Load PDF document
2. Split into text chunks
3. Convert chunks into embeddings
4. Store in FAISS vector database
5. Accept user query
6. Retrieve relevant chunks
7. Generate answer using LLM

## 1. Installing Required Libraries

In [ ]:
# Naye version ke hisaab se imports
# install karo agar pehle se nahi hai
# !pip install langchain langchain-community langchain-huggingface langchain-groq faiss-cpu pypdf python-dotenv langchain-text-splitters -q

## 2. Importing Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()
print('All libraries imported successfully')

All libraries imported successfully


## 3. Loading the PDF Document

We load our custom PDF using DirectoryLoader. This is the document our RAG system will use to answer questions.

In [ ]:
def load_pdf_file(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

# load the pdf
extracted_data = load_pdf_file(data='data/')

print(f'Total pages loaded : {len(extracted_data)}')
print(f'\nSample content from page 1 :')
print(extracted_data[0].page_content[:500])

Total pages loaded : 26

Sample content from page 1 :
AMAZON ML SUMMER SCHOOL
 2026 Master Preparation Guide
 All PYQs (2023-2025) + Predicted Paper + Python Solutions + Formula Sheet
 Compiled from: AMSS 2023, 2024, 2025 (Morning + Afternoon), Scaler Sample, Programming.md
 Video references: CampusX '100 Days of ML' + Code and Debug DSA in Python
TIME STRATEGY
MCQ Section
~2 min per question -- skip and return if stuck
Coding Section
Read both Qs first. Solve easier one fully first.
Stats/Probability
Write formula first, then substitute -- never j


## 4. Splitting Text into Chunks

We split the document into smaller chunks so that retrieval is more accurate. Each chunk is 500 characters with 20 characters of overlap between chunks to preserve context.

In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = text_split(extracted_data)

print(f'Total chunks created : {len(text_chunks)}')
print(f'\nSample Chunk 1 :')
print(text_chunks[0].page_content)
print(f'\nSample Chunk 2 :')
print(text_chunks[1].page_content)

Total chunks created : 135

Sample Chunk 1 :
AMAZON ML SUMMER SCHOOL
 2026 Master Preparation Guide
 All PYQs (2023-2025) + Predicted Paper + Python Solutions + Formula Sheet
 Compiled from: AMSS 2023, 2024, 2025 (Morning + Afternoon), Scaler Sample, Programming.md
 Video references: CampusX '100 Days of ML' + Code and Debug DSA in Python
TIME STRATEGY
MCQ Section
~2 min per question -- skip and return if stuck
Coding Section
Read both Qs first. Solve easier one fully first.
Stats/Probability

Sample Chunk 2 :
Stats/Probability
Write formula first, then substitute -- never jump
Linear Algebra
Matrix inverse: det first. If det=0, no inverse.
If stuck on MCQ
Eliminate wrong options -- usually 2 are obvious wrong
NOTE: The 2025 Morning Shift paper has date '03 Aug 2026' -- this is a predicted/compiled paper, not officially
released PYQ. Treat as high-quality practice, not 100% confirmed PYQ.


## 5. Creating Embeddings

We use HuggingFace's `sentence-transformers/all-MiniLM-L6-v2` model to convert each text chunk into a 384-dimensional vector. These vectors capture the semantic meaning of the text.

In [ ]:
def download_hugging_face_embeddings():
    # this model returns 384 dimensions
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

print('Loading HuggingFace embeddings model... (may take a minute on first run)')
embeddings = download_hugging_face_embeddings()
print('Embeddings model ready!')

# quick test
test_vector = embeddings.embed_query('test')
print(f'Embedding dimensions : {len(test_vector)}')

Loading HuggingFace embeddings model... (may take a minute on first run)
Embeddings model ready!
Embedding dimensions : 384


## 6. Building the FAISS Vector Store

FAISS (Facebook AI Similarity Search) is a local vector database. We store all chunk embeddings here so we can do fast similarity search when a user asks a question.

In [ ]:
print(f'Creating FAISS index for {len(text_chunks)} chunks...')
vectorstore = FAISS.from_documents(text_chunks, embeddings)
print('FAISS vector store created successfully!')

# save locally so we dont have to rebuild every time
vectorstore.save_local('faiss_index')
print('Vector store saved locally as faiss_index/')

Creating FAISS index for 135 chunks...
FAISS vector store created successfully!
Vector store saved locally as faiss_index/


## 7. Setting up the Retriever

The retriever does similarity search in the FAISS index. We use `k=3` which means for each query it will return the top 3 most relevant chunks.

In [ ]:
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})
print('Retriever setup done. Will return top 3 relevant chunks per query.')

Retriever setup done. Will return top 3 relevant chunks per query.


## 8. Setting up the LLM (Groq)

We use Groq's free API with the Llama 3.3 70B model as our language model. It generates the final answer using the retrieved context.

Set your Groq API key in a `.env` file: `GROQ_API_KEY = "your_key_here"`

Get a free key from: https://console.groq.com

In [ ]:
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.4,
    groq_api_key=GROQ_API_KEY
)

print('Groq LLM ready!')
print(f'Model : llama-3.3-70b-versatile')
print(f'Temperature : 0.4')

Groq LLM ready!
Model : llama-3.3-70b-versatile
Temperature : 0.4


## 9. Building the RAG Chain

This is the core of the system. The RAG chain:
1. Takes the user query
2. Retrieves relevant chunks from FAISS
3. Passes them as context to the LLM
4. LLM generates a grounded answer

In [ ]:
system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', '{input}'),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print('RAG chain built successfully!')

RAG chain built successfully!


## 10. Asking Questions — Showing Retrieved Chunks + Final Answer

Now we test our system. For each question we will show:
- The retrieved chunks (what the system found)
- The final generated answer

In [ ]:
def ask(question):
    print(f'Question : {question}')
    print('-' * 60)

    response = rag_chain.invoke({'input': question})

    # show retrieved chunks
    print('Retrieved Context Chunks :')
    for i, doc in enumerate(response['context']):
        print(f'\nChunk {i+1} (Page: {doc.metadata.get("page", "unknown")})')
        print(doc.page_content)

    print('\n' + '=' * 60)
    print(f'Final Answer : {response["answer"]}')
    print('=' * 60)
    print()

In [ ]:
ask('What is the main idea of the document?')

Question : What is the main idea of the document?
------------------------------------------------------------
Retrieved Context Chunks :

Chunk 1 (Page: 24)
TOPIC FREQUENCY ANALYSIS (How many times each topic appeared across all
papers)
Topic
2023
2024
2025A
2025M
Scaler
Total
Matrix Inverse [[1,2],[3,4]]
~
~
Yes
~
Yes
5+
Bayes Theorem
Yes
~
~
Yes
~
4
Variance / Covariance / Corr
Yes
~
Yes
~
Yes
4
PCA Properties
Yes
~
~
Yes
~
3
Bias-Variance Tradeoff
Yes
~
~
~
~
2+
K-Means Limitations
~
Yes
Yes
~
~
3
Gradient Descent vs Normal Eq
Yes
~
~
~
Yes
3
Eigenvalues
Yes
~
~
~
Yes
3
F1/Precision/Recall/FPR
~
Yes
~
~
Yes
3
Conditional Probability
Yes
Yes
Yes

Chunk 2 (Page: 25)
Good luck on your AMSS 2026 exam! Focus on what's in these papers -- don't start new
 topics.

Chunk 3 (Page: 12)
Explanation: Bayesian inference: Start with prior, observe data, compute posterior. Fundamentally about updating
beliefs.
Difficulty: Easy | Topic: Bayesian ML
CampusX: Video 85-86 (Bayes Theorem)
Q11. Valid a

In [ ]:
ask('What are the key topics covered in this document?')

Question : What are the key topics covered in this document?
------------------------------------------------------------
Retrieved Context Chunks :

Chunk 1 (Page: 24)
TOPIC FREQUENCY ANALYSIS (How many times each topic appeared across all
papers)
Topic
2023
2024
2025A
2025M
Scaler
Total
Matrix Inverse [[1,2],[3,4]]
~
~
Yes
~
Yes
5+
Bayes Theorem
Yes
~
~
Yes
~
4
Variance / Covariance / Corr
Yes
~
Yes
~
Yes
4
PCA Properties
Yes
~
~
Yes
~
3
Bias-Variance Tradeoff
Yes
~
~
~
~
2+
K-Means Limitations
~
Yes
Yes
~
~
3
Gradient Descent vs Normal Eq
Yes
~
~
~
Yes
3
Eigenvalues
Yes
~
~
~
Yes
3
F1/Precision/Recall/FPR
~
Yes
~
~
Yes
3
Conditional Probability
Yes
Yes
Yes

Chunk 2 (Page: 25)
Good luck on your AMSS 2026 exam! Focus on what's in these papers -- don't start new
 topics.

Chunk 3 (Page: 20)
Answer: C) Decision Tree (ID3/C4.5)
Explanation: ID3 uses Information Gain (based on entropy). CART uses Gini impurity. Both are decision tree
algorithms.
Predicted | Difficulty: Easy | Topic: Decis

In [ ]:
ask('Summarize the most important points from the document.')

Question : Summarize the most important points from the document.
------------------------------------------------------------
Retrieved Context Chunks :

Chunk 1 (Page: 25)
Good luck on your AMSS 2026 exam! Focus on what's in these papers -- don't start new
 topics.

Chunk 2 (Page: 24)
TOPIC FREQUENCY ANALYSIS (How many times each topic appeared across all
papers)
Topic
2023
2024
2025A
2025M
Scaler
Total
Matrix Inverse [[1,2],[3,4]]
~
~
Yes
~
Yes
5+
Bayes Theorem
Yes
~
~
Yes
~
4
Variance / Covariance / Corr
Yes
~
Yes
~
Yes
4
PCA Properties
Yes
~
~
Yes
~
3
Bias-Variance Tradeoff
Yes
~
~
~
~
2+
K-Means Limitations
~
Yes
Yes
~
~
3
Gradient Descent vs Normal Eq
Yes
~
~
~
Yes
3
Eigenvalues
Yes
~
~
~
Yes
3
F1/Precision/Recall/FPR
~
Yes
~
~
Yes
3
Conditional Probability
Yes
Yes
Yes

Chunk 3 (Page: 20)
Answer: C) Decision Tree (ID3/C4.5)
Explanation: ID3 uses Information Gain (based on entropy). CART uses Gini impurity. Both are decision tree
algorithms.
Predicted | Difficulty: Easy | Topic: 

## 11. Improvements and Experiments

The assignment asks us to try different settings and observe the impact.

### Experiment 1 — Different Chunk Sizes

Smaller chunks = more precise retrieval but less context per chunk.
Larger chunks = more context but retrieval may be less focused.

In [ ]:
# try with smaller chunk size
splitter_small = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_small = splitter_small.split_documents(extracted_data)

# try with larger chunk size
splitter_large = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks_large = splitter_large.split_documents(extracted_data)

print(f'Original chunk size 500 : {len(text_chunks)} chunks')
print(f'Smaller chunk size 200  : {len(chunks_small)} chunks')
print(f'Larger chunk size 1000  : {len(chunks_large)} chunks')
print('\nObservation: smaller chunks = more chunks = finer retrieval')

Original chunk size 500 : 135 chunks
Smaller chunk size 200  : 359 chunks
Larger chunk size 1000  : 70 chunks

Observation: smaller chunks = more chunks = finer retrieval


### Experiment 2 — Different k values for Retrieval

k=3 means we retrieve top 3 chunks. Increasing k gives more context but also more noise.

In [ ]:
# retriever with k=5 instead of k=3
retriever_k5 = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5})

question_answer_chain_k5 = create_stuff_documents_chain(llm, prompt)
rag_chain_k5 = create_retrieval_chain(retriever_k5, question_answer_chain_k5)

response_k5 = rag_chain_k5.invoke({'input': 'What is the main idea of the document?'})

print(f'With k=3 : retrieves 3 chunks')
print(f'With k=5 : retrieves 5 chunks')
print(f'\nAnswer with k=5 :')
print(response_k5['answer'])

With k=3 : retrieves 3 chunks
With k=5 : retrieves 5 chunks

Answer with k=5 :
The main idea of the document appears to be a study guide or review material for an exam, specifically the AMSS 2026 exam, covering various topics in machine learning and data analysis. The document provides topic frequency analysis, explanations, and practice questions to help students prepare for the exam.


### Experiment 3 — Effect of Temperature on Answer Style

Higher temperature = more creative answers. Lower temperature = more factual and focused answers.

In [ ]:
# low temperature - more factual
llm_low_temp = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.1, groq_api_key=GROQ_API_KEY)
chain_low = create_retrieval_chain(retriever, create_stuff_documents_chain(llm_low_temp, prompt))
response_low = chain_low.invoke({'input': 'What is the main idea of the document?'})

# high temperature - more creative
llm_high_temp = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.9, groq_api_key=GROQ_API_KEY)
chain_high = create_retrieval_chain(retriever, create_stuff_documents_chain(llm_high_temp, prompt))
response_high = chain_high.invoke({'input': 'What is the main idea of the document?'})

print('Temperature = 0.1 (Factual) :')
print(response_low['answer'])
print('\nTemperature = 0.9 (Creative) :')
print(response_high['answer'])

Temperature = 0.1 (Factual) :
The main idea of the document appears to be a study guide or review for an exam, specifically the AMSS 2026 exam, focusing on key topics in machine learning and statistics. The document provides a frequency analysis of topics and highlights important concepts to focus on. It also includes explanations and practice questions to help with exam preparation.

Temperature = 0.9 (Creative) :
The main idea of the document is to analyze the frequency of various topics in academic papers from 2023 to 2025, likely to help students prepare for the AMSS 2026 exam. The document provides a summary of key topics and their appearance across different papers. It also includes explanations and answers to specific questions related to machine learning and statistics.


## 12. Key Learnings

- **RAG combines retrieval and generation** — the LLM alone cannot answer questions about private documents, but with RAG it can
- **Chunk size matters** — too small and you lose context, too large and retrieval becomes noisy
- **Embeddings capture semantic meaning** — similar meaning chunks are stored close together in vector space
- **FAISS is efficient** — local vector similarity search is fast even with hundreds of chunks
- **Temperature controls creativity** — lower temperature gives more consistent factual answers which is better for document QA
- **k value controls context** — more retrieved chunks gives more context but also more chance of irrelevant information

## Conclusion

This RAG system successfully demonstrates how to build a pipeline that understands user queries, retrieves relevant information from a custom document, and generates accurate answers. RAG systems are widely used in chatbots, knowledge assistants, enterprise search systems, and AI-powered documentation tools.